## Step 7 - ASR extraction (faster-whisper), one JSON per video

Transcribes the **spoken audio** of every video with **faster-whisper** (`vi`)
and writes **one JSON file per video** so the dataset is trivial to extend: when
new videos arrive, re-run and only the missing videos are processed.

```
ASR/<video_name>.json     # [ {"start_ms": 0, "end_ms": 5000, "text": "..."}, ... ]
                          # e.g. ASR/L21_V001.json
```

`video_name = <batch>_<video_id>` (e.g. `L21_V001`) - the same join key the
backend's `asr_service` uses, so the per-video files merge straight into
`data/asr/asr_data.json` (last cell) with no remapping. Each video maps to a
**list of timestamped segments** `{start_ms, end_ms, text}`, exactly the shape
`asr_service.search_asr` expects.

**Resumable per video** - a video whose `ASR/<video_name>.json` already exists is
skipped entirely. Transcription is atomic per video (Whisper emits all segments
in one decode), so the natural checkpoint unit is the whole video; the file is
written via a temp-then-rename so a killed session never leaves a half JSON.

**Uses both GPUs** - on Kaggle's "GPU T4 x2" we load one model per card and
transcribe videos in parallel (one video per GPU), ~2x throughput. Falls back to
a single CPU model when no GPU is visible.

Input is the **original `.mp4` videos** (faster-whisper decodes the audio track
directly via PyAV - no separate ffmpeg extraction needed). Set Accelerator =
**"GPU T4 x2"** for the dual-GPU speedup.

> Why faster-whisper: 4-5x faster than openai-whisper at the same accuracy,
> runs on CTranslate2 (int8/float16), strong Vietnamese support, and a single
> `pip install` with no kernel restart on Kaggle's stock environment.

In [ ]:
# faster-whisper pulls in ctranslate2 + av (PyAV). The RAPIDS "pip's dependency
# resolver" warnings about dask-cuda / cuml / cudf are unrelated and safe to ignore.
!pip -q install faster-whisper

In [ ]:
import os
import json
import threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import ctranslate2
from tqdm.auto import tqdm
from faster_whisper import WhisperModel

In [ ]:
# Original videos. Step 2 reads them from Videos_<batch>/video/*.mp4; on Kaggle
# upload them as a dataset and point VIDEOS_DIR at it. We discover *.mp4
# recursively, so any nesting works as long as the filename stem is the
# video_name (e.g. L21_V001.mp4 -> L21_V001).
VIDEOS_DIR = '/kaggle/input/aic-videos'
OUTPUT_DIR = './ASR'              # one <video_name>.json per video
VIDEO_EXTS = ('.mp4', '.mkv', '.webm', '.avi', '.mov')

LANGUAGE   = 'vi'                 # force Vietnamese (skip per-video lang detection)
MODEL_SIZE = 'large-v3'          # best vi accuracy; drop to 'medium' if VRAM-tight

# transcribe tuning for noisy broadcast audio (music beds, overlapping speech).
TRANSCRIBE_KW = dict(
    language=LANGUAGE,
    beam_size=5,
    vad_filter=True,            # drop non-speech (music/silence) -> fewer hallucinations
    vad_parameters=dict(min_silence_duration_ms=500),
    condition_on_previous_text=False,  # stops runaway repetition on broadcast audio
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# One model PER GPU (loads weights once each). Kaggle's "GPU T4 x2" gives two
# devices; we run them in parallel below (one video per GPU at a time). Pinning a
# WhisperModel to device_index=i keeps each transcribe call on its own card.
# Falls back to a single CPU int8 model if no CUDA device is visible.
N_GPU = ctranslate2.get_cuda_device_count()

if N_GPU > 0:
    models = [
        WhisperModel(MODEL_SIZE, device='cuda', device_index=i, compute_type='float16')
        for i in range(N_GPU)
    ]
    print(f'faster-whisper ready on {N_GPU} GPU(s) (float16), '
          f'model = {MODEL_SIZE}, lang = {LANGUAGE}')
else:
    models = [WhisperModel(MODEL_SIZE, device='cpu', compute_type='int8')]
    print(f'no GPU visible -> 1 CPU model (int8). For speed set Accelerator to '
          f'"GPU T4 x2". model = {MODEL_SIZE}, lang = {LANGUAGE}')

In [ ]:
def asr_video(model, path):
    """Transcribe one video -> [ {start_ms, end_ms, text}, ... ] in time order.

    model.transcribe is lazy (returns a generator); iterating it runs the decode.
    `model` is passed in so each worker thread uses its own GPU's model.
    """
    segments, _info = model.transcribe(str(path), **TRANSCRIBE_KW)
    out = []
    for seg in segments:
        text = (seg.text or '').strip()
        if not text:
            continue
        out.append({
            'start_ms': int(seg.start * 1000),
            'end_ms': int(seg.end * 1000),
            'text': text,
        })
    return out

### Debug a single video (optional)

Run this on one video to sanity-check the transcript + segment timing before the
full sweep. Empty / garbled output -> the audio may be music-only (fine to skip)
or try `MODEL_SIZE='medium'` and loosen `vad_parameters`.

In [ ]:
dbg_path = f'{VIDEOS_DIR}/Videos_L21/video/L21_V001.mp4'  # <- a video to inspect
for s in asr_video(models[0], dbg_path)[:10]:
    print(f"[{s['start_ms']/1000:6.1f}-{s['end_ms']/1000:6.1f}s]  {s['text']}")

In [ ]:
def out_path(video_name):
    return Path(OUTPUT_DIR) / f'{video_name}.json'

def save_video_asr(video_name, segments):
    # temp-then-rename: an interrupted write never leaves a half-written JSON that
    # the per-video skip would later mistake for a completed video.
    p = out_path(video_name)
    tmp = p.with_suffix('.json.tmp')
    with tmp.open('w', encoding='utf-8') as f:
        json.dump(segments, f, ensure_ascii=False, indent=2)
    tmp.replace(p)

In [ ]:
# Discover every video file: {video_name: path}. The filename stem is the
# video_name join key (e.g. L21_V001.mp4 -> L21_V001).
#
# Guard trùng tên (như step 1/2): gộp theo TÊN FILE, không theo folder. Vài
# dataset upload lặp một part (Videos_L25_a vs Videos_L25_a1 cùng dải id):
#   * cùng tên + CÙNG dung lượng -> bản sao, bỏ qua (giữ bản gặp đầu).
#   * cùng tên + KHÁC dung lượng -> hai video khác nội dung xài chung id -> DỪNG.
root = Path(VIDEOS_DIR)
videos = {}
seen_size = {}
redundant = 0
conflicts = []
for vid in root.rglob('*'):
    if vid.is_file() and vid.suffix.lower() in VIDEO_EXTS:
        if vid.stem in videos:
            if vid.stat().st_size == seen_size[vid.stem]:
                redundant += 1
            else:
                conflicts.append((vid.stem, str(videos[vid.stem]), str(vid)))
            continue
        videos[vid.stem] = vid    # stem == L21_V001
        seen_size[vid.stem] = vid.stat().st_size

print('videos found:', len(videos))
if redundant:
    print(f'[WARN] bỏ qua {redundant} file trùng-lặp (cùng tên & cùng dung lượng, '
          'vd Videos_L25_a vs Videos_L25_a1) -> giữ bản gặp đầu tiên.')
if conflicts:
    print(f'[LỖI] {len(conflicts)} tên video trùng NHƯNG khác nội dung:')
    for name, a, b in conflicts[:20]:
        print(f'  {name}: {a}  <->  {b}')
    raise SystemExit(
        f'{len(conflicts)} video khác nội dung dùng chung tên. Xử lý trùng trước '
        'khi transcribe (xem danh sách trên).'
    )


In [ ]:
# Parallel sweep: one worker thread per GPU model. CTranslate2 releases the GIL
# during decode, so the threads run their cards concurrently. Each video writes
# its own JSON, so there are no shared-write races; only tqdm is guarded.
todo = [v for v in sorted(videos) if not out_path(v).exists()]
print(f'{len(todo)} videos to transcribe on {len(models)} model(s) '
      f'({len(videos) - len(todo)} already done)')

pbar = tqdm(total=len(todo), desc='videos')
lock = threading.Lock()

def run_chunk(model, names):
    """Transcribe this worker's slice of videos on its dedicated model."""
    for video_name in names:
        try:
            segments = asr_video(model, videos[video_name])
        except Exception as e:
            print('WARN', videos[video_name], '->', e)
            segments = []
        save_video_asr(video_name, segments)       # write even if empty -> won't retry
        with lock:
            pbar.update(1)

# Round-robin so the slices are length-balanced regardless of per-video duration.
chunks = [todo[i::len(models)] for i in range(len(models))]
with ThreadPoolExecutor(max_workers=len(models)) as ex:
    futures = [ex.submit(run_chunk, m, c) for m, c in zip(models, chunks)]
    for f in futures:
        f.result()                                 # re-raise any worker exception
pbar.close()

print('done. per-video JSON in', OUTPUT_DIR)

### Merge into the backend's single `asr_data.json` (optional)

The backend's `asr_service` reads one flat dict `{video_name: [segments]}`. The
per-video files merge into it by a plain `dict` build, so you can rebuild
`asr_data.json` any time without re-running ASR - and adding a new video only
means dropping in its JSON and re-merging. Copy the result to
`backend/data/asr/asr_data.json`.

In [ ]:
merged = {}
for jf in sorted(Path(OUTPUT_DIR).glob('*.json')):
    with jf.open(encoding='utf-8') as f:
        merged[jf.stem] = json.load(f)

with open('asr_data.json', 'w', encoding='utf-8') as f:
    json.dump(merged, f, ensure_ascii=False, indent=2)
print('merged videos:', len(merged),
      '| segments:', sum(len(v) for v in merged.values()), '-> asr_data.json')